In [50]:
import nvdlib
import json
import pandas as pd
import numpy as np
import re
import ast
from tqdm.notebook import tqdm
import sys
import string
import nltk
from nltk.tokenize import word_tokenize
import os
import fnmatch
import warnings
from pprint import pprint

from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt')

stopwords = list(filter(lambda x: len(x)>1, stopwords.words('english')))
punctuation = set(string.punctuation)
punctuation.remove('(')
punctuation.remove(')')
punctuation.remove('$')

warnings.filterwarnings('ignore')

[nltk_data] Downloading package stopwords to /home/umd-
[nltk_data]     user/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/umd-user/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [394]:
def getFiles(description):
    # extensions = ['php', 'html', 'js']
    extensions = ['java',"jsp","html","js"]
    result = []
    for ext in extensions:
        result += re.findall(r'\b\w+\.' + ext + r'\b', description, re.IGNORECASE)
    return result

def getVersions(description):
    result = re.findall(r'\d+\.\d+\.\d+', description)
    for version in result: description = description.replace(version, '')
    result += re.findall(r'\d+\.\d+', description)
    if result==[]:
        return ['0']
    else: return result

def getVulnerability(description):
    desc = description.lower()
    if "sql" in desc:
        return "SQL Injection"
    elif "xss" in desc or "cross-site scripting" in desc or "cross site scripting" in desc:
        return "XSS"
    # elif "file upload" in desc or "file inclusion" in desc:
    #     return "File Inclusion"
    # elif "file access" in desc:
    #     return "File Access"
    # elif "session" in desc:
    #     return "Session Fixation"
    # elif "code injection" in desc:
    #     return "Code Injection"
    # elif "command" in desc:
    #     return "Command Execution"
    # elif "csrf" in desc or "request forgery" in desc:
    #     return "CSRF"
    else:
        return "NA"
    
# Compute whether appVersion was released before cveVersion (inclusive)
def compareVersions(appVersion, cveVersions):
    flag = False
    if (len(cveVersions)==0 or len(appVersion)==0 or cveVersions==['0']): flag = True
    for cveVersion in cveVersions:
        v1 = list(map(int, appVersion.split('.')))
        v2 = list(map(int, cveVersion.split('.')))
        size = min(len(v1), len(v2))
        v1 = v1[:size]
        v2 = v2[:size]
        if (v2 <= v1): 
            # print(v1,v2)
            flag = True
    return flag

def getParameters(description):
    words = word_tokenize(description)
    words = [word for word in words if word.lower() not in stopwords and word not in punctuation]
    indices = [i for i in range(1, len(words)) if words[i] == "parameter" or words[i] == "parameters"]
    params = [words[i-1] for i in indices]
    indices = [i for i in range(1, len(words)-2) if (words[i-1]=='(' and words[i].isdigit() and words[i+1]==')')]
    params += [words[i+2] for i in indices]
    indices = [i for i in range(len(words)-1) if words[i] == "function" or words[i]=="$"]
    params += [words[i+1] for i in indices]
    params = [param.replace('"', '') for param in params if ('.java' not in param and '/' not in param)]
    return list(set(params))

def getCVEFromNavex(cve, file):
    flag = False
    for index, row in cve.iterrows():
        for fileName in row['filenames']:
            if fileName in file:
                flag = True
                cve_id = row['id']
                break
    if not flag: cve_id = 'NA'
    return cve_id

def filterOnFiles(fileNames):
    if fileNames == []: return True
    for root, dirs, files in os.walk(directory_path):
        for fileName in fileNames:
            for filename in fnmatch.filter(files, fileName):
                # file_path = os.path.join(root, filename)
                return True
    return False

def substrAsIdentifier(main_string, substring):
    pattern = re.compile(f'{re.escape(substring)}(?![a-zA-Z0-9])')
    return bool(pattern.search(main_string))

def getFilesFromParameters(parameters):
    file_paths = []
    # pattern = r'(?:' + '|'.join(re.escape(parameter) for parameter in parameters) + r')\b'
    if parameters == []: return file_paths
    for root, dirs, files in os.walk(directory_path):
        for filename in files:
            file_path = os.path.join(root, filename)
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
                content = file.read()
                for parameter in parameters:
                    if substrAsIdentifier(content, parameter):
                        file_paths.append(file_path)
                        break
    return list(set(file_paths))

In [500]:
tqdm.pandas()
directory_path = "/home/umd-user/joern/projectcpg/apps/java_apps/openfire"
appName ="openfire"
extension = ''
appVersion = '5'

In [501]:
r = nvdlib.searchCVE(keywordSearch=appName)
jsonFormattedCVE = json.dumps(ast.literal_eval(str(r)))
cve_ld = pd.read_json(jsonFormattedCVE)

In [502]:
cve=cve_ld.copy()

In [503]:
cve = pd.read_json(jsonFormattedCVE)

cve['descriptions'] = cve['descriptions'].apply(lambda descriptions: list(filter(lambda x: x["lang"]=="en", descriptions)))
cve['descriptions'] = cve['descriptions'].apply(lambda descriptions: descriptions[0]['value'])
cve['versions'] = cve['descriptions'].apply(getVersions)

cve['filenames'] = cve['descriptions'].apply(getFiles)
cve['cve_vulnerability'] = cve['descriptions'].apply(getVulnerability)
cve['parameters'] = cve['descriptions'].apply(getParameters)
cve['relevant_version'] = cve['versions'].apply(lambda x: compareVersions(appVersion, x))# and compareVersions(x[0], ['6.0']))

cve = cve[cve['cve_vulnerability']!='NA']
cve = cve[cve['relevant_version']==True]

cve = cve[cve['filenames'].apply(filterOnFiles)]
# cve = cve[cve['parameters'].progress_apply(lambda params: getFilesFromParameters(params) != [])]
# cve = cve[cve['filenames'].map(lambda x: x!=[]) | cve['parameters'].map(lambda x: x!=[])]
# cve['filenames'] = cve.progress_apply(lambda x: x.filenames if x.filenames!=[] else list(map(lambda x: x.split('/')[-1], getFilesFromParameters(x.parameters))), axis=1)
# cve = cve[cve['parameters'].apply(lambda params: params != ["query"] and isinstance(params, list))]
cve = cve.loc[:, ['id', 'cve_vulnerability', 'versions', 'filenames', 'parameters', 'descriptions']]

cve.to_excel(f"/home/umd-user/joern/projectcpg/cve/{appName}-{extension}cve.xlsx")
cve.shape

(6, 6)

In [504]:
cve

,id,cve_vulnerability,versions,filenames,parameters,descriptions
21,CVE-2018-11688,XSS,[3.9.2],[],[],Ignite Realtime Openfire before 3.9.2 is vulne...
22,CVE-2019-15488,XSS,[4.4.1],[],[],Ignite Realtime Openfire before 4.4.1 has refl...
25,CVE-2019-20363,XSS,[4.4.4],[],[],An XSS issue was discovered in Ignite Realtime...
27,CVE-2019-20365,XSS,[4.4.4],[],[],An XSS issue was discovered in Ignite Realtime...
28,CVE-2019-20366,XSS,[4.4.4],[],[],An XSS issue was discovered in Ignite Realtime...
34,CVE-2020-24602,XSS,[4.5.1],[],[GET],Ignite Realtime Openfire 4.5.1 has a reflected...


In [309]:
# cve = cve[cve["descriptions"].str.contains(r'Jenkins [a-z]', na=False, regex=True)]


In [427]:
full_text = "".join(cve["descriptions"])
print(full_text)

SQL injection vulnerability in CallLogDAO in SIP Plugin in Openfire 3.6.0a and earlier allows remote attackers to execute arbitrary SQL commands via the type parameter to sipark-log-summary.jsp.Cross-site scripting (XSS) vulnerability in login.jsp in the Admin Console in Openfire 3.6.0a and earlier allows remote attackers to inject arbitrary web script or HTML via the url parameter.Cross-site scripting (XSS) vulnerability in the login form (login.jsp) of the admin console in Openfire (formerly Wildfire) 2.2.2, and possibly other versions before 2.3.0 Beta 2, allows remote attackers to inject arbitrary web script or HTML via the username parameter, a different vulnerability than CVE-2005-4877.Cross-site scripting (XSS) vulnerability in the login form (login.jsp) of the admin console in Openfire (formerly Wildfire) 2.3.0 Beta 2 allows remote attackers to inject arbitrary web script or HTML via Javascript events in the username parameter, a different vulnerability than CVE-2005-4876.Cross

In [477]:
cve

,id,cve_vulnerability,versions,filenames,parameters,descriptions
1,CVE-2008-6509,SQL Injection,[3.6.0],[summary.jsp],[type],SQL injection vulnerability in CallLogDAO in S...
2,CVE-2008-6510,XSS,[3.6.0],[login.jsp],[url],Cross-site scripting (XSS) vulnerability in lo...
4,CVE-2005-4876,XSS,"[2.2.2, 2.3.0]",[login.jsp],[username],Cross-site scripting (XSS) vulnerability in th...
5,CVE-2005-4877,XSS,[2.3.0],[login.jsp],[username],Cross-site scripting (XSS) vulnerability in th...
6,CVE-2006-7233,XSS,"[2.6.0, 3.5.3]",[login.jsp],[url],Cross-site scripting (XSS) vulnerability in th...
9,CVE-2009-0496,XSS,[3.6.2],"[logviewer.jsp, log.jsp, summary.jsp, properti...","[roomconfig_roomdesc, maxTotalSize, maxDays, r...",Multiple cross-site scripting (XSS) vulnerabil...
14,CVE-2015-6972,XSS,[3.10.2],"[bookmark.jsp, bookmark.jsp, details.jsp, summ...","[search, groupchatName, hostname, urlName]",Multiple cross-site scripting (XSS) vulnerabil...
19,CVE-2017-15911,XSS,[4.1.7],[settings.jsp],[],The Admin Console in Ignite Realtime Openfire ...
26,CVE-2019-20364,XSS,[4.4.4],[SystemCacheDetails.jsp],[],An XSS issue was discovered in Ignite Realtime...
29,CVE-2019-20528,XSS,[4.4.1],[standard.jsp],[username],Ignite Realtime Openfire 4.4.1 allows XSS via ...


In [529]:
# Output of the navex joern extension
appName = 'simplify'
navex = pd.read_json(f'/home/umd-user/joern/projectcpg/paths/{appName}-{extension}output.json')
navex

,pathid,vulnerability,nodeid,methodname,filename,linenumber,code,sanitized
0,1,SQL Injection,29726,main,simplify/smalivm/src/main/java/org/cf/smalivm/...,44,String args,FALSE
1,1,SQL Injection,29738,main,simplify/smalivm/src/main/java/org/cf/smalivm/...,46,args,FALSE
2,1,SQL Injection,29737,main,simplify/smalivm/src/main/java/org/cf/smalivm/...,46,args[1],FALSE
3,1,SQL Injection,29736,main,simplify/smalivm/src/main/java/org/cf/smalivm/...,46,className,FALSE
4,1,SQL Injection,29758,main,simplify/smalivm/src/main/java/org/cf/smalivm/...,48,className,FALSE
...,...,...,...,...,...,...,...,...
572,11,XSS,31653,main,simplify/smalivm/src/main/java/org/cf/smalivm/...,53,args[0],FALSE
573,11,XSS,31647,main,simplify/smalivm/src/main/java/org/cf/smalivm/...,53,"FRAMEWORK_ROOT + ""/"" + args[0]",FALSE
574,11,XSS,31646,main,simplify/smalivm/src/main/java/org/cf/smalivm/...,53,resPath,FALSE
575,11,XSS,31662,main,simplify/smalivm/src/main/java/org/cf/smalivm/...,54,resPath,FALSE


In [530]:
navex["pathid"].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16])

In [531]:
len(navex["pathid"].unique())

16

In [509]:
    /home/umd-user/joern/joern --script /home/umd-user/joern/projectcpg/src/RunNavex.scala   --import /home/umd-user/joern/projectcpg/src/main/php/old_version/Constants.scala   --import /home/umd-user/joern/projectcpg/src/main/php/old_version/SQLConstants.scala      --import /home/umd-user/joern/projectcpg/src/main/php/old_version/SanitizationFilter.scala   --import /home/umd-user/joern/projectcpg/src/main/php/old_version/DBConstraint.scala   --import /home/umd-user/joern/projectcpg/src/main/php/old_version/Utils.scala   --import /home/umd-user/joern/projectcpg/src/main/php/old_version/NavexMain.scala --param path="/home/umd-user/joern/projectcpg/apps/php-stats-0.1.9.2"

SyntaxError: invalid syntax (3830565172.py, line 1)

In [510]:
import pandas as pd

# Define variables
output_file = f'/home/umd-user/joern/projectcpg/paths/{appName}-{extension}filtered.json'


# Filter rows with pathid >= 12
filtered_navex = navex[navex["pathid"] >= 12]

# Save the filtered DataFrame back to JSON
filtered_navex.to_json(output_file, orient='records', lines=True)


In [511]:
filtered_data = filtered_navex.to_dict(orient='records')  # Convert DataFrame back to list of dictionaries

# Save filtered data to JSON file
with open(output_file, 'w') as file:
    json.dump(filtered_data, file, indent=4)

In [512]:
list(navex["code"][navex["pathid"]==1])

['getParameter("key")',
 'key',
 'key',
 'String name',
 'name',
 'Object key',
 'key',
 'this.properties.get(key)',
 'return properties.get(key);',
 'public String get(Object key)',
 'this.sessions.get(sessionid)',
 'session',
 'session',
 'session',
 'session',
 'session',
 'session',
 'session',
 'getParameter("key")',
 'key',
 'key',
 'String name',
 'name',
 'Object key',
 'key',
 'this.properties.get(key)',
 'return properties.get(key);',
 'public String get(Object key)',
 'this.sessionMap.get(streamID)',
 'return sessionMap.get(streamID);',
 'public HttpSession getSession(String streamID)',
 'getSession()',
 'session',
 'session',
 'flash',
 'flash',
 'request']

In [513]:
import ast
import numpy as np

def getCVEidsFromPath(row, pathRow):
    """
    Match CVE identifiers based on filenames and parameters.

    Args:
        row (dict): CVE row with 'filenames', 'parameters', 'cve_vulnerability', and 'id'.
        pathRow (dict): Path row with 'filename', 'vulnerability', 'code', and 'methodname'.

    Returns:
        str or NaN: CVE ID if a match is found, otherwise NaN.
    """
    flag = False
    cve_id = np.nan

    # Parse 'filenames' and 'parameters' if stored as strings
    files = row['filenames']
    if type(files) != list:
        files = ast.literal_eval(files)

    cveParams = row['parameters']
    if type(cveParams) != list:
        cveParams = ast.literal_eval(cveParams)

    # Default to empty list if no files or parameters
    if not files:
        files = ['']
    if not cveParams:
        cveParams = []

    # Iterate over filenames
    for fileName in files:
        if row['cve_vulnerability'] == pathRow['vulnerability']:
            # Check if fileName matches pathRow's filename
            if not cveParams and fileName.lower() in pathRow['filename'].lower():
                flag = True
            else:
                # Check parameters in code or as method name
                for param in cveParams:
                    if (
                        fileName.lower() in pathRow['filename'].lower() and 
                        (substrAsIdentifier(pathRow['code'], param) or param == pathRow['methodname'])
                    ):
                        flag = True
            if flag:
                cve_id = row['id']
                break  # Stop once a match is found

    return cve_id

In [514]:
CVE_ids = navex.progress_apply(lambda navexRow: list(cve.apply(lambda x: getCVEidsFromPath(x, navexRow), axis=1).dropna().unique()), axis=1)
navex['CVE_ids'] = CVE_ids
navex.head() 

  0%|          | 0/3140 [00:00<?, ?it/s]

,pathid,vulnerability,nodeid,methodname,filename,linenumber,code,sanitized,CVE_ids
0,1,SQL Injection,23141,encryptProperty,Openfire/xmppserver/src/main/java/org/jivesoft...,163,"getParameter(""key"")",FALSE,[]
1,1,SQL Injection,23140,encryptProperty,Openfire/xmppserver/src/main/java/org/jivesoft...,163,key,FALSE,[]
2,1,SQL Injection,23148,encryptProperty,Openfire/xmppserver/src/main/java/org/jivesoft...,164,key,FALSE,[]
3,1,SQL Injection,421702,getProperty,Openfire/xmppserver/src/main/java/org/jivesoft...,524,String name,FALSE,[]
4,1,SQL Injection,421727,getProperty,Openfire/xmppserver/src/main/java/org/jivesoft...,531,name,FALSE,[]


In [515]:
# Check rows that matched with a CVE
emptyList = pd.Series([np.nan] * len(navex['CVE_ids'])).fillna('[]')
navex[navex['CVE_ids'].astype(str) != emptyList].explode('CVE_ids').head()

,pathid,vulnerability,nodeid,methodname,filename,linenumber,code,sanitized,CVE_ids
153,1,XSS,23141,encryptProperty,Openfire/xmppserver/src/main/java/org/jivesoft...,163,"getParameter(""key"")",FALSE,CVE-2018-11688
153,1,XSS,23141,encryptProperty,Openfire/xmppserver/src/main/java/org/jivesoft...,163,"getParameter(""key"")",FALSE,CVE-2019-15488
153,1,XSS,23141,encryptProperty,Openfire/xmppserver/src/main/java/org/jivesoft...,163,"getParameter(""key"")",FALSE,CVE-2019-20363
153,1,XSS,23141,encryptProperty,Openfire/xmppserver/src/main/java/org/jivesoft...,163,"getParameter(""key"")",FALSE,CVE-2019-20365
153,1,XSS,23141,encryptProperty,Openfire/xmppserver/src/main/java/org/jivesoft...,163,"getParameter(""key"")",FALSE,CVE-2019-20366


In [516]:
identified=navex[navex['CVE_ids'].astype(str) != emptyList].explode('CVE_ids')

In [517]:
len(identified["pathid"].unique())

184

In [518]:
uknow_paths=list(set(navex["pathid"].unique()).difference(set(identified["pathid"].unique())))[:20]

In [519]:
path_tocheck=[]
for id_path in uknow_paths:
    path_tocheck.append(list(navex["code"][navex["pathid"]==id_path]))

In [520]:
import json


# Convert the list of lists to JSON format
paths_json = json.dumps(path_tocheck, indent=4)

# Save the JSON to a file
with open('paths.json', 'w') as json_file:
    json_file.write(paths_json)


In [521]:
navex = navex.explode('CVE_ids')
navex.shape

(15089, 9)

In [522]:
navex = navex.merge(cve, how='left', left_on='CVE_ids', right_on='id').drop(columns=['id', 'cve_vulnerability'])

In [523]:
matchedCVEs = {e for e in navex['CVE_ids'].dropna()}
print("Number of exploit matches:", len(matchedCVEs), "out of #" + str(len(cve['id'])), "CVEs")
pprint(matchedCVEs)

Number of exploit matches: 6 out of #6 CVEs
{'CVE-2018-11688',
 'CVE-2019-15488',
 'CVE-2019-20363',
 'CVE-2019-20365',
 'CVE-2019-20366',
 'CVE-2020-24602'}


In [471]:
# Save data to excel
dfToSave = navex.dropna()
dfToSave.reset_index(inplace=True)
dfToSave.to_excel(f"/home/umd-user/joern/projectcpg/cve/{appName}-{extension}navex.xlsx")
dfToSave.shape

(14936, 14)

In [20]:
xssPaths = navex[navex['vulnerability'] == 'XSS']
# xssPaths = navex
idsAcrossDb = xssPaths[xssPaths.apply(lambda x: 'query' in x['code'], 1)]['pathid'].unique()
acrossDb = xssPaths[xssPaths.apply(lambda x: x['pathid'] in idsAcrossDb, 1)]
notDb = xssPaths[xssPaths['pathid'].apply(lambda x: x not in acrossDb['pathid'])]
set(acrossDb['CVE_ids'].unique()) - set(notDb['CVE_ids'].unique())

set()

In [21]:

acrossDb['CVE_ids'].unique().shape, notDb['CVE_ids'].unique().shape

((0,), (1,))

In [22]:
navex.shape

(5963, 13)

In [23]:
navex

,pathid,vulnerability,nodeid,methodname,filename,linenumber,code,sanitized,CVE_ids,versions,filenames,parameters,descriptions
0,1,SQL Injection,443908,getName,liquibase-master/liquibase-standard/src/main/j...,25,"this.getAttribute(""name"", String.class)",FALSE,NaN,NaN,NaN,NaN,NaN
1,1,SQL Injection,443907,,liquibase-master/liquibase-standard/src/main/j...,25,"return getAttribute(""name"", String.class);",NA,NaN,NaN,NaN,NaN,NaN
2,1,SQL Injection,443905,,liquibase-master/liquibase-standard/src/main/j...,23,public String getName(),NA,NaN,NaN,NaN,NaN,NaN
3,1,SQL Injection,353710,get,liquibase-master/liquibase-standard/src/main/j...,239,getName(),FALSE,NaN,NaN,NaN,NaN,NaN
4,1,SQL Injection,353706,get,liquibase-master/liquibase-standard/src/main/j...,239,(LiquibaseSerializable) object.getSerializable...,FALSE,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5958,333,XSS,256907,alreadyExists,liquibase-master/liquibase-standard/src/main/j...,131,backingIndex.getName(),FALSE,CVE-2020-2283,[1.4.5],[],[],Jenkins Liquibase Runner Plugin 1.4.5 and earl...
5959,333,XSS,441361,<init>,liquibase-master/liquibase-standard/src/main/j...,31,String indexName,FALSE,CVE-2020-2283,[1.4.5],[],[],Jenkins Liquibase Runner Plugin 1.4.5 and earl...
5960,333,XSS,441368,<init>,liquibase-master/liquibase-standard/src/main/j...,33,indexName,FALSE,CVE-2020-2283,[1.4.5],[],[],Jenkins Liquibase Runner Plugin 1.4.5 and earl...
5961,333,XSS,441427,setName,liquibase-master/liquibase-standard/src/main/j...,55,String name,FALSE,CVE-2020-2283,[1.4.5],[],[],Jenkins Liquibase Runner Plugin 1.4.5 and earl...


In [ ]:
import os
import subprocess
import shutil

# List of repository URLs
repos = [
    "https://github.com/macrozheng/mall",
    "https://github.com/spring-projects/spring-boot",
    "https://github.com/kdn251/interviews",
    "https://github.com/dromara/hutool",
    "https://github.com/xuxueli/xxl-job",
    "https://github.com/apache/flink",
    "https://github.com/alibaba/Sentinel",
    "https://github.com/PowerJob/PowerJob",
    "https://github.com/wiremock/wiremock",
    "https://github.com/liyifeng1994/ssm",
    "https://github.com/liquibase/liquibase",
    "https://github.com/puniverse/quasar",
    "https://github.com/CalebFenton/simplify",
    "https://github.com/igniterealtime/Openfire",
    "https://github.com/testng-team/testng",
    "https://github.com/zalando/logbook",
    "https://github.com/vaadin/framework",
    "https://github.com/spotify/genie",
    "https://github.com/nashtech-garage/yas",
    "https://github.com/Qbian61/forum-java",
    "https://github.com/fuzhengwei/interview",
    "https://github.com/pgjdbc/pgjdbc",
    "https://github.com/puniverse/capsule",
    "https://github.com/PebbleTemplates/pebble",
    "https://github.com/Zephery/newblog",
    "https://github.com/sakaiproject/sakai",
    "https://github.com/TIBCOSoftware/jasperreports",
    "https://github.com/objectionary/eo",
    "https://github.com/datavane/tis",
    "https://github.com/repeats/Repeat",
    "https://github.com/protegeproject/protege",
    "https://github.com/yahoo/elide",
    "https://github.com/OpenNMS/opennms",
    "https://github.com/twitter/hbc",
    "https://github.com/jfaster/mango",
    "https://github.com/bardsoftware/ganttproject",
    "https://github.com/payara/Payara",
    "https://github.com/superblaubeere27/obfuscator",
    "https://github.com/Erudika/scoold",
    "https://github.com/esig/dss",
    "https://github.com/RohitAwate/Everest",
    "https://github.com/steve-community/steve"
]

# Directory to clone repositories into
clone_dir = "/home/umd-user/joern/projectcpg/apps/"

def get_directory_size(path):
    """Calculate the size of a directory in MB."""
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if os.path.isfile(fp):
                total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)  # Convert bytes to MB

for repo in repos:
    repo_name = repo.split("/")[-1]
    repo_path = os.path.join(clone_dir, repo_name)
    
    # Clone the repository
    if not(os.path.isdir(repo_path)):
        print(f"Cloning {repo}...")

        try:
            subprocess.run(["git", "clone", repo, repo_path], check=True)
        except subprocess.CalledProcessError as e:
            print(f"Failed to clone {repo}: {e}")
            continue
        
        # Check the size of the cloned repository
        repo_size = get_directory_size(repo_path)
        print(f"Size of {repo_name}: {repo_size:.2f} MB")
        
        # Delete the repository if it exceeds 15 MB
        if repo_size < 15 and repo_size<50:
            print(f"{repo_name} exceeds 15 MB. Deleting...")
            shutil.rmtree(repo_path)

print("Cloning process completed. Only repositories <= 15 MB are retained.")


In [ ]:
import os
import subprocess

# Paths
base_clone_dir = "/home/umd-user/joern/projectcpg/apps"
output_dir = "/home/umd-user/joern/projectcpg/apps"

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# Iterate over directories in the cloned_repos folder
for repo_name in os.listdir(base_clone_dir):
    repo_path = os.path.join(base_clone_dir, repo_name)
    output_path = os.path.join(output_dir, f"{repo_name}.bin")
    
    # Check if it is a directory
    if os.path.isdir(repo_path):
        # Skip if the .bin file already exists
        if os.path.exists(output_path):
            print(f"Output file {output_path} already exists. Skipping {repo_name}.")
            continue

        print(f"Processing repository: {repo_name}")
        
        # Construct the joern-parse command
        command = f"sh /home/umd-user/joern/joern-parse {repo_path} -o {output_path}"
        
        # Execute the command
        try:
            os.system(command)
            print(f"Successfully processed {repo_name}. Output saved to {output_path}.")
        except Exception as e:
            print(f"Failed to process {repo_name}: {e}")


In [ ]:
import requests

def fetch_github_repos(topic, language, max_repos=1000):
    base_url = "https://api.github.com/search/repositories"
    headers = {"Accept": "application/vnd.github+json"}
    repos = []
    per_page = 100  # Maximum allowed by GitHub API
    pages = (max_repos // per_page) + (1 if max_repos % per_page != 0 else 0)

    for page in range(1, pages + 1):
        params = {
            "q": f"topic:{topic} language:{language}",
            "sort": "stars",
            "order": "desc",
            "per_page": per_page,
            "page": page
        }
        response = requests.get(base_url, headers=headers, params=params)

        if response.status_code == 200:
            data = response.json()
            repos.extend(
                [
                    {
                        "name": repo["name"],
                        "url": repo["html_url"],
                        "stars": repo["stargazers_count"],
                    }
                    for repo in data.get("items", [])
                ]
            )
            if len(data.get("items", [])) < per_page:
                break  # No more repositories
        else:
            print(f"Error: {response.status_code}, {response.json()}")
            break

    return repos[:max_repos]


if __name__ == "__main__":
    topic = "java"
    language = "java"
    max_repos = 1000  # Adjust as needed

    print(f"Fetching top {max_repos} repositories for topic '{topic}' and language '{language}'...")
    repositories = fetch_github_repos(topic, language, max_repos)

    for idx, repo in enumerate(repositories, 1):
        print(f"{idx}. {repo['name']} ({repo['stars']} stars) - {repo['url']}")


In [ ]:
tqdm.pandas()
directory_path = "/home/umd-user/joern/projectcpg/apps/spring-boot-main"
for idx, repo in enumerate(repositories, 1):
    
    appName =repo["name"]
    extension = ''
    appVersion = '0'
    r = nvdlib.searchCVE(keywordSearch=appName)
    jsonFormattedCVE = json.dumps(ast.literal_eval(str(r)))
    cve = pd.read_json(jsonFormattedCVE)
    cve = pd.read_json(jsonFormattedCVE)
    if cve.shape!=(0,0):
        cve['descriptions'] = cve['descriptions'].apply(lambda descriptions: list(filter(lambda x: x["lang"]=="en", descriptions)))
        cve['descriptions'] = cve['descriptions'].apply(lambda descriptions: descriptions[0]['value'])
        cve['versions'] = cve['descriptions'].apply(getVersions)
        cve['filenames'] = cve['descriptions'].apply(getFiles)
        cve['cve_vulnerability'] = cve['descriptions'].apply(getVulnerability)
        cve['parameters'] = cve['descriptions'].apply(getParameters)
        cve['relevant_version'] = cve['versions'].apply(lambda x: compareVersions(appVersion, x))# and compareVersions(x[0], ['6.0']))

        cve = cve[cve['cve_vulnerability']!='NA']
        cve = cve[cve['relevant_version']==True]

        #cve = cve[cve['filenames'].apply(filterOnFiles)]
        #cve = cve[cve['parameters'].progress_apply(lambda params: getFilesFromParameters(params) != [])]
        #cve = cve[cve['filenames'].map(lambda x: x!=[]) | cve['parameters'].map(lambda x: x!=[])]
        #cve['filenames'] = cve.progress_apply(lambda x: x.filenames if x.filenames!=[] else list(map(lambda x: x.split('/')[-1], getFilesFromParameters(x.parameters))), axis=1)
        #cve = cve[cve['parameters'].apply(lambda params: params != ["query"] and isinstance(params, list))]
        cve = cve.loc[:, ['id', 'cve_vulnerability', 'versions', 'filenames', 'parameters', 'descriptions']]

        cve.to_excel(f"/home/umd-user/joern/projectcpg/cve/{appName}-{extension}cve.xlsx")
        if cve.shape[0]>0:
            print(repo["name"],repo["url"] )
            print(cve.shape)

In [443]:
import requests

def fetch_github_repos(topic, language, max_repos=1000, target_repos=None):
    base_url = "https://api.github.com/search/repositories"
    headers = {"Accept": "application/vnd.github+json"}
    repos = []
    per_page = 100  # Maximum allowed by GitHub API
    pages = (max_repos // per_page) + (1 if max_repos % per_page != 0 else 0)

    for page in range(1, pages + 1):
        params = {
            "q": f"topic:{topic} language:{language}",
            "sort": "stars",
            "order": "desc",
            "per_page": per_page,
            "page": page
        }
        response = requests.get(base_url, headers=headers, params=params)

        if response.status_code == 200:
            data = response.json()
            repos.extend(
                [
                    {
                        "name": repo["name"],
                        "url": repo["html_url"],
                        "stars": repo["stargazers_count"],
                    }
                    for repo in data.get("items", [])
                ]
            )
            if len(data.get("items", [])) < per_page:
                break  # No more repositories
        else:
            print(f"Error: {response.status_code}, {response.json()}")
            break

    # Filter for the target repositories
    if target_repos:
        repos = [repo for repo in repos if repo['name'].lower() in target_repos]

    return repos[:max_repos]


if __name__ == "__main__":
    topic = "java"
    language = "java"
    max_repos = 1000  # Adjust as needed
    target_repos = {
        "dss", "eo", "forum-java", "gantproject", "jasperreports",
        "logbook", "mango", "liquibase", "obfuscator", "jenkins",
        "openfire", "scoold", "sentinel", "simplify"
    }  # Set of target repository names

    print(f"Fetching top {max_repos} repositories for topic '{topic}' and language '{language}'...")
    repositories = fetch_github_repos(topic, language, max_repos, target_repos)

    for idx, repo in enumerate(repositories, 1):
        print(f"{idx}. {repo['name']} ({repo['stars']} stars) - {repo['url']}")


Fetching top 1000 repositories for topic 'java' and language 'java'...
1. jenkins (23410 stars) - https://github.com/jenkinsci/jenkins
2. Sentinel (22512 stars) - https://github.com/alibaba/Sentinel
3. liquibase (4813 stars) - https://github.com/liquibase/liquibase
4. simplify (4479 stars) - https://github.com/CalebFenton/simplify
5. Openfire (2885 stars) - https://github.com/igniterealtime/Openfire
6. logbook (1870 stars) - https://github.com/zalando/logbook
7. forum-java (1634 stars) - https://github.com/Qbian61/forum-java
8. jasperreports (1080 stars) - https://github.com/TIBCOSoftware/jasperreports
9. eo (1071 stars) - https://github.com/objectionary/eo
10. mango (919 stars) - https://github.com/jfaster/mango
11. obfuscator (882 stars) - https://github.com/superblaubeere27/obfuscator
12. scoold (873 stars) - https://github.com/Erudika/scoold
13. dss (835 stars) - https://github.com/esig/dss
